This notebook is made to replicate how we found our data.

In [149]:
import pandas as pd
import requests

def get_census_data(url, variables, state_fips='06'): #change the fips code to change the state
    """Helper function to fetch data from the Census API."""
    params = {
        'get': ",".join(variables),
        'for': 'tract:*',
        'in': f'state:{state_fips} county:*'
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API Request failed with status code {response.status_code}: {response.text}")
        
    data = response.json()
    return pd.DataFrame(data[1:], columns=data[0])

# --- 1. Define Endpoints and Variables ---
# 2020 Decennial Census PL 94-171 Redistricting Data (Corrected Endpoint)
url_2020 = "https://api.census.gov/data/2020/dec/pl"
vars_2020 = ['P1_001N', 'P2_002N', 'P1_003N', 'P1_004N', 'P1_008N']

# 2022 ACS 5-Year Estimates Detailed Tables
url_2022 = "https://api.census.gov/data/2022/acs/acs5"
vars_2022 = [
    'B03002_001E', 'B03002_012E', 'B02001_002E', 'B02001_003E', 'B02001_007E', 
    'B04004_022E', 'B04004_045E', 'B04004_074E', 'B04004_097E', 'B04004_016E',
    'B04004_082E', 
    'B04004_030E', 'B04004_007E', 'B04004_048E', 'B04004_008E', 'B04004_050E', 
    'B04004_009E', 'B04004_010E', 'B04004_056E', 'B04004_011E', 'B04004_012E', 
    'B04004_084E', 'B04004_013E', 'B04004_091E', 'B04004_006E', 'B04004_014E', 
    'B04004_015E', 'B04004_017E', 'B02008_001E', 'B02009_001E', 'B02011_001E',
    'B02013_001E'
]

# --- 2. Fetch the Data ---
print("Fetching 2020 Decennial PL data...")
df_2020 = get_census_data(url_2020, vars_2020)

print("Fetching 2022 ACS 5-Year data...")
df_2022 = get_census_data(url_2022, vars_2022)

# --- 3. Merge and Clean Data ---
df_merged = pd.merge(df_2020, df_2022, on=['state', 'county', 'tract'])

# Create the standard 11-digit GEOID
df_merged['GEOID'] = df_merged['state'] + df_merged['county'] + df_merged['tract']

# Reorder columns to match the requested output
requested_cols = ['state', 'county', 'tract', 'GEOID'] + vars_2020 + vars_2022
df_final = df_merged[requested_cols].copy()

# Convert the data columns to numeric types 
data_cols = vars_2020 + vars_2022
df_final[data_cols] = df_final[data_cols].apply(pd.to_numeric, errors='coerce')

print("Dataframe creation complete!")
print(df_final.head())

Fetching 2020 Decennial PL data...
Fetching 2022 ACS 5-Year data...
Dataframe creation complete!
  state county   tract        GEOID  P1_001N  P2_002N  P1_003N  P1_004N  \
0    06    001  400100  06001400100     3038      205     1870      144   
1    06    001  400200  06001400200     2001      207     1421       40   
2    06    001  400300  06001400300     5504      547     3413      561   
3    06    001  400400  06001400400     4112      374     2765      289   
4    06    001  400500  06001400500     3644      437     1957      625   

   P1_008N  B03002_001E  ...  B04004_013E  B04004_091E  B04004_006E  \
0       80         3269  ...            0           25           80   
1       41         2147  ...            0            5           43   
2      191         5619  ...            0            0           42   
3      122         4278  ...            0            0            7   
4      168         3949  ...            0            0            0   

   B04004_014E  B04004_01

In [150]:
def make_mena(df, name, subset_list):
    for ancestry in subset_list: 
        df[name] = df[name] + df[ancestry]

In [151]:
mena_world_bank = ['B04004_007E', 'B04004_048E', 'B04004_008E', 'B04004_050E', 
                   'B04004_009E', 'B04004_010E', 'B04004_056E', 'B04004_011E', 
                   'B04004_012E', 'B04004_013E', 'B04004_014E']
mena_unhcr = ['B04004_007E', 'B04004_008E', 'B04004_009E', 'B04004_056E', 
              'B04004_011E', 'B04004_012E', 'B04004_013E', 'B04004_014E']

mena_unsd = ['B04004_016E', 'B04004_030E', 'B04004_007E', 'B04004_008E', 'B04004_050E', 
             'B04004_009E', 'B04004_010E','B04004_011E', 'B04004_012E', 'B04004_084E', 
             'B04004_013E', 'B04004_091E', 'B04004_014E', 'B04004_015E','B04004_017E']
mena_government = ['B04004_014E', 'B04004_015E', 'B04004_017E', 'B04004_091E', 'B04004_050E', 'B04004_048E',
                   'B04004_013E', 'B04004_084E', 'B04004_082E', 'B04004_012E', 'B04004_011E',
                   'B04004_010E', 'B04004_009E', 'B04004_008E', 'B04004_007E']

In [152]:
df_final['mena_world_bank_ACS'] = 0
df_final['mena_unhcr_ACS'] = 0
df_final['mena_unsd_ACS'] = 0
df_final['mena_government_ACS'] = 0

In [153]:
make_mena(df_final, 'mena_world_bank_ACS', mena_world_bank)

make_mena(df_final, 'mena_unhcr_ACS', mena_unhcr)
make_mena(df_final, 'mena_unsd_ACS', mena_unsd)

make_mena(df_final, 'mena_government_ACS', mena_government)

In [154]:
df_final

,state,county,tract,GEOID,P1_001N,P2_002N,P1_003N,P1_004N,P1_008N,B03002_001E,...,B04004_015E,B04004_017E,B02008_001E,B02009_001E,B02011_001E,B02013_001E,mena_world_bank_ACS,mena_unhcr_ACS,mena_unsd_ACS,mena_government_ACS
0,06,001,400100,06001400100,3038,205,1870,144,80,3269,...,0,0,2566,201,595,79,189,80,105,214
1,06,001,400200,06001400200,2001,207,1421,40,41,2147,...,43,0,1824,110,359,144,0,0,48,48
2,06,001,400300,06001400300,5504,547,3413,561,191,5619,...,42,0,4005,596,997,586,0,0,42,42
3,06,001,400400,06001400400,4112,374,2765,289,122,4278,...,0,0,3266,571,591,258,17,0,7,17
4,06,001,400500,06001400500,3644,437,1957,625,168,3949,...,0,0,2067,1098,641,471,0,0,13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,06115040902,2412,430,1572,214,72,1868,...,0,0,1477,276,137,88,0,0,0,0
9125,06,115,041001,06115041001,3594,626,2623,26,358,3672,...,0,0,3292,77,15,619,0,0,0,0
9126,06,115,041002,06115041002,3935,387,3225,22,147,3417,...,0,0,3260,47,42,389,31,0,31,31
9127,06,115,041101,06115041101,3187,254,2552,33,86,2288,...,0,3,2200,134,74,58,0,0,3,3


We cannot divide by 0!

In [156]:
df_final = df_final[df_final['P1_001N']!= 0]
df_final = df_final[df_final['B03002_001E'] !=0]

In [157]:
def make_percentage_columns(df, subset_pop, total_pop):
    df[subset_pop + '_pct'] = df[subset_pop]/df[total_pop]

In [158]:
pl_cols = ['P2_002N', 'P1_003N', 'P1_004N', 'P1_008N']
for col in pl_cols:
    make_percentage_columns(df_final, col, 'P1_001N')

In [159]:
acs_cols = ['B03002_012E', 'B02001_002E', 'B02001_003E', 'B02001_007E', 
    'B04004_022E', 'B04004_045E', 'B04004_074E', 'B04004_097E', 'B04004_016E', 
    'B04004_030E', 'B04004_007E', 'B04004_048E', 'B04004_008E', 'B04004_050E', 
    'B04004_009E', 'B04004_010E', 'B04004_056E', 'B04004_011E', 'B04004_012E', 
    'B04004_084E', 'B04004_013E', 'B04004_091E', 'B04004_006E', 'B04004_014E', 
    'B04004_015E', 'B04004_017E', 'B02008_001E', 'B02009_001E', 'B02011_001E',
    'B02013_001E'
]
for col in acs_cols:
    make_percentage_columns(df_final, col, 'B03002_001E')

In [160]:
mena_cols = ['mena_world_bank_ACS', 'mena_unhcr_ACS', 'mena_unsd_ACS', 'mena_government_ACS']
for col in mena_cols:
    make_percentage_columns(df_final, col, 'B03002_001E')

In [161]:
print(df_final.columns)

Index(['state', 'county', 'tract', 'GEOID', 'P1_001N', 'P2_002N', 'P1_003N',
       'P1_004N', 'P1_008N', 'B03002_001E', 'B03002_012E', 'B02001_002E',
       'B02001_003E', 'B02001_007E', 'B04004_022E', 'B04004_045E',
       'B04004_074E', 'B04004_097E', 'B04004_016E', 'B04004_082E',
       'B04004_030E', 'B04004_007E', 'B04004_048E', 'B04004_008E',
       'B04004_050E', 'B04004_009E', 'B04004_010E', 'B04004_056E',
       'B04004_011E', 'B04004_012E', 'B04004_084E', 'B04004_013E',
       'B04004_091E', 'B04004_006E', 'B04004_014E', 'B04004_015E',
       'B04004_017E', 'B02008_001E', 'B02009_001E', 'B02011_001E',
       'B02013_001E', 'mena_world_bank_ACS', 'mena_unhcr_ACS', 'mena_unsd_ACS',
       'mena_government_ACS', 'P2_002N_pct', 'P1_003N_pct', 'P1_004N_pct',
       'P1_008N_pct', 'B03002_012E_pct', 'B02001_002E_pct', 'B02001_003E_pct',
       'B02001_007E_pct', 'B04004_022E_pct', 'B04004_045E_pct',
       'B04004_074E_pct', 'B04004_097E_pct', 'B04004_016E_pct',
       'B04004_030

In [162]:
percent_dictionary = {
    'P2_002N_pct': 'hispanic_PL_pct',
    'P1_003N_pct': 'white_PL_pct',
    'P1_004N_pct': 'black_PL_pct',
    'P1_008N_pct': 'sor_PL_pct',
    'B03002_012E_pct': 'hispanic_ACS_pct',
    'B02001_002E_pct': 'white_ACS_pct',
    'B02001_003E_pct': 'black_ACS_pct',
    'B02001_007E_pct': 'sor_ACS_pct',
    'B04004_022E_pct': 'brazilian_ACS_pct',
    'B04004_045E_pct': 'guyanese_ACS_pct',
    'B04004_074E_pct': 'cabo_verdean_ACS_pct',
    'B04004_097E_pct': 'belizean_ACS_pct',
    'B04004_016E_pct': 'armenian_ACS_pct',
    'B04004_030E_pct': 'cypriot_ACS_pct',
    'B04004_007E_pct': 'egyptian_ACS_pct',
    'B04004_048E_pct': 'iranian_ACS_pct',
    'B04004_008E_pct': 'iraqi_ACS_pct',
    'B04004_050E_pct': 'israeli_ACS_pct',
    'B04004_009E_pct': 'jordanian_ACS_pct',
    'B04004_010E_pct': 'lebanese_ACS_pct',
    'B04004_056E_pct': 'maltese_ACS_pct',
    'B04004_011E_pct': 'moroccan_ACS_pct',
    'B04004_012E_pct': 'palestinian_ACS_pct',
    'B04004_084E_pct': 'sudanese_ACS_pct',
    'B04004_013E_pct': 'syrian_ACS_pct', 
    'B04004_091E_pct': 'turkish_ACS_pct',
    'B04004_006E_pct': 'arab_ACS_pct',
    'B04004_014E_pct': 'arab_arab_ACS_pct',
    'B04004_015E_pct': 'arab_other_ACS_pct',
    'B04004_017E_pct': 'chaldean_ACS_pct',
    'B02008_001E_pct': 'white_alone_or_combo_ACS_pct',
    'B02009_001E_pct': 'black_alone_or_combo_ACS_pct',
    'B02011_001E_pct': 'asian_alone_or_combo_ACS_pct',
    'B02013_001E_pct': 'sor_alone_or_combo_ACS_pct'}

In [163]:
df_final.rename(columns=percent_dictionary, inplace=True)

In [164]:
print(df_final.columns)

Index(['state', 'county', 'tract', 'GEOID', 'P1_001N', 'P2_002N', 'P1_003N',
       'P1_004N', 'P1_008N', 'B03002_001E', 'B03002_012E', 'B02001_002E',
       'B02001_003E', 'B02001_007E', 'B04004_022E', 'B04004_045E',
       'B04004_074E', 'B04004_097E', 'B04004_016E', 'B04004_082E',
       'B04004_030E', 'B04004_007E', 'B04004_048E', 'B04004_008E',
       'B04004_050E', 'B04004_009E', 'B04004_010E', 'B04004_056E',
       'B04004_011E', 'B04004_012E', 'B04004_084E', 'B04004_013E',
       'B04004_091E', 'B04004_006E', 'B04004_014E', 'B04004_015E',
       'B04004_017E', 'B02008_001E', 'B02009_001E', 'B02011_001E',
       'B02013_001E', 'mena_world_bank_ACS', 'mena_unhcr_ACS', 'mena_unsd_ACS',
       'mena_government_ACS', 'hispanic_PL_pct', 'white_PL_pct',
       'black_PL_pct', 'sor_PL_pct', 'hispanic_ACS_pct', 'white_ACS_pct',
       'black_ACS_pct', 'sor_ACS_pct', 'brazilian_ACS_pct', 'guyanese_ACS_pct',
       'cabo_verdean_ACS_pct', 'belizean_ACS_pct', 'armenian_ACS_pct',
       'cy

In [165]:
df_final

,state,county,tract,GEOID,P1_001N,P2_002N,P1_003N,P1_004N,P1_008N,B03002_001E,...,arab_other_ACS_pct,chaldean_ACS_pct,white_alone_or_combo_ACS_pct,black_alone_or_combo_ACS_pct,asian_alone_or_combo_ACS_pct,sor_alone_or_combo_ACS_pct,mena_world_bank_ACS_pct,mena_unhcr_ACS_pct,mena_unsd_ACS_pct,mena_government_ACS_pct
0,06,001,400100,06001400100,3038,205,1870,144,80,3269,...,0.000000,0.000000,0.784950,0.061487,0.182013,0.024166,0.057816,0.024472,0.032120,0.065463
1,06,001,400200,06001400200,2001,207,1421,40,41,2147,...,0.020028,0.000000,0.849558,0.051234,0.167210,0.067070,0.000000,0.000000,0.022357,0.022357
2,06,001,400300,06001400300,5504,547,3413,561,191,5619,...,0.007475,0.000000,0.712760,0.106069,0.177434,0.104289,0.000000,0.000000,0.007475,0.007475
3,06,001,400400,06001400400,4112,374,2765,289,122,4278,...,0.000000,0.000000,0.763441,0.133474,0.138149,0.060309,0.003974,0.000000,0.001636,0.003974
4,06,001,400500,06001400500,3644,437,1957,625,168,3949,...,0.000000,0.000000,0.523424,0.278045,0.162320,0.119271,0.000000,0.000000,0.003292,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,06115040902,2412,430,1572,214,72,1868,...,0.000000,0.000000,0.790685,0.147752,0.073340,0.047109,0.000000,0.000000,0.000000,0.000000
9125,06,115,041001,06115041001,3594,626,2623,26,358,3672,...,0.000000,0.000000,0.896514,0.020969,0.004085,0.168573,0.000000,0.000000,0.000000,0.000000
9126,06,115,041002,06115041002,3935,387,3225,22,147,3417,...,0.000000,0.000000,0.954053,0.013755,0.012291,0.113843,0.009072,0.000000,0.009072,0.009072
9127,06,115,041101,06115041101,3187,254,2552,33,86,2288,...,0.000000,0.001311,0.961538,0.058566,0.032343,0.025350,0.000000,0.000000,0.001311,0.001311


In [182]:
df_final['P1_003N'].sum()/df_final['P1_001N'].sum()

np.float64(0.5012577643566228)

In [36]:
df_final['mena_government_ACS_pct'].max()

0.21333333333333335

In [75]:
df_final['white_alone_or_combo_ACS_pct'].mean()

0.6496414657310939

In [182]:
df_final.to_csv("06acs_pl_govmena.csv")